In [14]:
#Library
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from collections import Counter
import sys
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from preprocessing import preprocessing


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder



from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import spacy.cli
spacy.cli.download("en_core_web_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 98.9 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [15]:
df_train=pd.read_csv("data/train.csv")
df_test=pd.read_csv("data/test.csv")
df_train=df_train.drop(columns=['id'])
df_train=df_train.drop_duplicates()

In [16]:
ponctuation = [".", "!", "?"]
preprocesser=FunctionTransformer(preprocessing)

In [17]:
X_train=df_train.drop(columns="target")
y_train=df_train["target"]

In [18]:
columns=["has_.","has_?","has_!","has_url","has_CAP","url_is_https"]

features = ColumnTransformer([
    ("txt", TfidfVectorizer(), "texte_clean"),                    # ← la colonne nettoyée
    ("kw",  OneHotEncoder(handle_unknown="ignore"), ["keyword"]),
    ("binaire", "passthrough", columns),
])

pipe = Pipeline([
    ("preprocesser", preprocesser),   # nettoyage seulement
    ("features", features),             # remplace l'étape "vect"
    ("clf", LogisticRegression(max_iter=1000)),
])


param_grid = [
    {"features": [CountVectorizer(), TfidfVectorizer()],
     "features__ngram_range": [(1, 1), (1, 2)],
     "features__min_df": [1, 3]},
]

In [19]:
pipe.fit(X_train.head(50), y_train.head(50))

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocesser', ...), ('features', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['keyword','location','text']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function pre...x7fe9badc0f40>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False


In [20]:
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro",n_jobs=-1)

In [21]:
print("X_train :", X_train.shape)
out = pipe.named_steps["preprocesser"].transform(X_train)
print("après preprocessing :", out.shape, type(out))

X_train : (7561, 3)
après preprocessing : (7561, 10) <class 'pandas.DataFrame'>


In [22]:
print(out.columns.tolist())

['keyword', 'text', 'has_.', 'has_!', 'has_?', 'has_url', 'url_is_https', 'has_CAP', 'tokens', 'texte_clean']


In [ ]:
grid.fit(X_train,y_train)

In [ ]:
y_test_pred=grid.predict(df_test)

submission = pd.DataFrame({
    "id":df_test["id"],
    "target":y_test_pred
})

submission.to_csv("submission_pipeline.csv",index=False)
